# Python Modules & Imports
Covers: import, from...import, as alias, __name__, relative/absolute imports, sys.path, sys.modules, circular imports, lazy imports, __all__, importlib

## 1. Basic import Statement

In [ ]:
import math
import os
import sys

print('math.pi:', math.pi)
print('os.getcwd():', os.getcwd())
print('Python version:', sys.version[:6])

## 2. from...import — Import Specific Names

In [ ]:
from math import sqrt, pi, ceil, floor
from os.path import join, exists, basename

print('sqrt(25):', sqrt(25))
print('pi:', pi)
print('ceil(3.2):', ceil(3.2))
print('join result:', join('/usr', 'local', 'bin'))
print('basename:', basename('/usr/local/bin/python'))

## 3. as Alias — Rename Imports

In [ ]:
import os.path as osp
import collections as col
from datetime import datetime as dt
from collections import defaultdict as dd

print('osp.sep:', osp.sep)
print('dt.now():', dt.now())

# defaultdict with alias
word_count = dd(int)
for word in ['apple', 'banana', 'apple', 'cherry', 'banana', 'apple']:
    word_count[word] += 1
print('word_count:', dict(word_count))

## 4. __name__ == '__main__' Pattern

In [ ]:
# In a notebook, __name__ is '__main__'
# In an imported module, __name__ is the module's filename

print('Current __name__:', __name__)

# Simulate what happens in a module file:
# ----------------------------------------
# def greet(name):
#     return f'Hello, {name}!'
#
# if __name__ == '__main__':
#     # This block ONLY runs when the file is executed directly
#     # NOT when imported by another module
#     print(greet('World'))
# ----------------------------------------

# Demonstration using exec
module_code = '''
def greet(name):
    return f"Hello, {name}!"

if __name__ == "__main__":
    print("Running as script:", greet("World"))
else:
    print(f"Imported as module: {__name__}")
'''

# When __name__ is __main__ (direct execution)
exec(compile(module_code, '<string>', 'exec'), {'__name__': '__main__'})

# When imported (different __name__)
exec(compile(module_code, '<string>', 'exec'), {'__name__': 'mymodule'})

## 5. sys.path — Module Search Path

In [ ]:
import sys

print('sys.path entries:')
for i, path in enumerate(sys.path[:5]):
    print(f'  [{i}] {path}')

print('\nTotal entries:', len(sys.path))

# You can add paths at runtime
# sys.path.insert(0, '/path/to/my/modules')  # highest priority
# sys.path.append('/path/to/other/modules')  # lowest priority

# Check where a module was loaded from
import json
print('\njson module file:', json.__file__)

## 6. sys.modules — Module Cache

In [ ]:
import sys

# Check if a module is already imported
print('math in sys.modules:', 'math' in sys.modules)
print('numpy in sys.modules:', 'numpy' in sys.modules)

# Get cached module
import math
cached_math = sys.modules['math']
print('\nCached math module:', cached_math)
print('Same object?', cached_math is math)  # True — same object

# Count loaded modules
print('\nTotal loaded modules:', len(sys.modules))

# Show some loaded modules
loaded = [name for name in sys.modules if not name.startswith('_')][:10]
print('Sample loaded modules:', loaded)

## 7. Circular Import — Demonstration and Solutions

In [ ]:
import sys
import types

# Simulate circular import problem and solution
# In real code, circular imports happen across files

# PROBLEM: Module A imports B, Module B imports A
# This causes ImportError or partially-initialized module issues

# SOLUTION 1: Lazy import (import inside function)
def solution_lazy_import():
    """Import inside function defers until call time."""
    import json  # deferred import
    return json.dumps({'solution': 'lazy import'})

print('Lazy import result:', solution_lazy_import())

# SOLUTION 2: Import module, not name
import os  # import module object
# Use os.path.join instead of from os.path import join
# This works even if os is partially initialized

# SOLUTION 3: TYPE_CHECKING guard
from typing import TYPE_CHECKING
if TYPE_CHECKING:
    # This import only happens during static analysis (mypy)
    # NOT at runtime — breaks circular dependency
    pass  # from mypackage.models import User

print('Circular import solutions demonstrated')

## 8. Lazy Imports — Defer Expensive Imports

In [ ]:
import time

# Pattern 1: Import inside function
def process_data_lazy():
    """Heavy imports only happen when this function is called."""
    import json       # lightweight, but demonstrates the pattern
    import hashlib
    data = {'key': 'value'}
    serialized = json.dumps(data)
    return hashlib.md5(serialized.encode()).hexdigest()

print('Lazy result:', process_data_lazy())

# Pattern 2: importlib for dynamic/optional imports
import importlib

def try_import(module_name, fallback=None):
    """Try to import a module, return fallback if not available."""
    try:
        return importlib.import_module(module_name)
    except ImportError:
        return fallback

ujson = try_import('ujson')  # fast JSON library
json_lib = ujson if ujson else try_import('json')
print('JSON library:', json_lib.__name__)

# Pattern 3: Check before importing
spec = importlib.util.find_spec('numpy')
if spec is not None:
    print('numpy is available')
else:
    print('numpy is NOT installed')

## 9. __all__ — Controlling Public API

In [ ]:
# Demonstrate __all__ behavior using a temporary module
import sys
import types

# Create a module dynamically to demonstrate __all__
module_code = '''
__all__ = ['PublicClass', 'public_function']

class PublicClass:
    """This is part of the public API."""
    def method(self):
        return "public"

def public_function():
    """This is part of the public API."""
    return "public function"

def _private_function():
    """This is NOT in __all__ — internal use only."""
    return "private"

class _InternalClass:
    """Not exported."""
    pass
'''

# Create and register the module
mod = types.ModuleType('demo_module')
exec(module_code, mod.__dict__)
sys.modules['demo_module'] = mod

import demo_module
print('__all__:', demo_module.__all__)
print('PublicClass:', demo_module.PublicClass)
print('public_function:', demo_module.public_function())
print('_private_function still accessible:', demo_module._private_function())
print('\nNote: __all__ only restricts "from module import *"')
print('Direct access to private names still works')

## 10. importlib — Dynamic Imports

In [ ]:
import importlib
import importlib.util

# 1. Import by string name
module_name = 'json'
json_mod = importlib.import_module(module_name)
result = json_mod.dumps({'dynamic': True, 'import': 'works'})
print('Dynamic import result:', result)

# 2. Import submodule
os_path = importlib.import_module('os.path')
print('os.path.sep:', os_path.sep)

# 3. Check if module exists without importing
for mod_name in ['json', 'numpy', 'pandas', 'nonexistent_module']:
    spec = importlib.util.find_spec(mod_name)
    status = 'available' if spec else 'NOT found'
    print(f'  {mod_name}: {status}')

# 4. Reload a module (useful in development)
import math
math_reloaded = importlib.reload(math)
print('\nReloaded math module:', math_reloaded is math)  # True — same object, re-executed

## 11. Interview Q&A Summary

In [ ]:
# Q1: Difference between import module vs from module import name
import math as math_module
from math import pi as pi_value

# Modifying module attribute
original_pi = math_module.pi
math_module.pi = 999

print('math_module.pi after modification:', math_module.pi)  # 999
print('pi_value (local reference):', pi_value)               # unchanged!

# Restore
math_module.pi = original_pi
print('Restored math_module.pi:', math_module.pi)

In [ ]:
# Q2: Demonstrate __name__ behavior
print('In notebook, __name__ is:', __name__)  # __main__

# Q3: sys.path search order
import sys
print('\nFirst 3 sys.path entries:')
for p in sys.path[:3]:
    print(f'  {repr(p)}')

# Q4: Check for circular import risk
# Best practice: keep imports at top level, use lazy imports for circular deps

# Q5: __all__ best practice
# Always define __all__ in library modules to document public API

# Q6: Absolute vs relative — PEP 8 recommends absolute imports
# from mypackage.models import User  # absolute (preferred)
# from .models import User           # relative (OK within packages)

print('\nKey takeaways:')
print('1. import module vs from module import name — namespace and rebinding differ')
print('2. __name__ == __main__ prevents side effects on import')
print('3. sys.path controls where Python looks for modules')
print('4. sys.modules caches imported modules')
print('5. Circular imports: use lazy imports or restructure')
print('6. __all__ defines public API for star imports')
print('7. importlib enables dynamic/programmatic imports')